# Build your poker bot

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/G1-100/bot-showdown-3d/blob/main/examples/colab_starter.ipynb)

Write **`act(state)`**, export **`poker_bot.py`**, and submit it through your host's invitation link. Everything you need to write and check the starter is in this notebook; no repository clone or GitHub account is needed.

1. **Save your own copy:** in Colab choose **File → Save a copy in Drive**. Sign in to your Google account if prompted, then work in the new copy. This keeps your edits in your Drive.
2. **Edit your algorithm:** find **1. Write your bot** below and change `act(state)`. Claude Code can help you write it; paste the resulting Python into that cell, keeping its first `%%writefile` line.
3. **Run the bot cell, then the checks:** click the play button beside each code cell (or press Shift+Enter), working from top to bottom. Connect the standard Python runtime when prompted; this starter needs no GPU or package installs. Re-run the bot cell after every edit.
4. **Export your file:** run **3. Download your submission** to download `poker_bot.py` to your computer. Local checks help catch mistakes; the submission server performs the authoritative validation.
5. **Join your host's event:** open the student invitation link supplied by your host, upload `poker_bot.py`, and follow the validation status. Keep your returned update key privately so you can submit revisions.

The host's server runs the tournament after submission. You do not need Docker, a web server, deployment access, or changes to the poker engine. Submit only the exported `.py` file; your notebook stays in Drive. Use the host's shared invitation rather than a `localhost` address.


## 1. Write your bot

Keep `%%writefile poker_bot.py` as the first line of the next cell. Running the cell writes only its remaining contents into the submission file. Re-run it after every edit.

Start with the baseline below or replace it with code you develop in Claude Code. You can add ordinary top-level helper functions. Do not put notebook test or download code in this cell.

**Prompt for Claude Code:**

> Improve my poker bot's act(state) function. Return one dictionary with action equal to fold, check, call, or raise; a raise also includes an integer raiseTo. Use only my holeCards and public state, respect canRaise/minRaiseTo/maxRaiseTo, and fit within a 500 ms decision budget. Keep everything in one Python file, with no classes, annotations, decorators, introspection, dynamic execution, files, network, external packages, AI services, trained models, or imported poker engines. The only permitted imports are approved functions from math, random, itertools, and collections. Explain the strategy, then give me the source to paste into my Colab writefile cell. Do not change or run the tournament server.

The server enforces a restricted Python subset. Those four libraries are an allowlist, not permission to use every Python feature. Use only ordinary functions, loops, comprehensions, arithmetic, and supported collection methods. Keep the source under 20 KB; compute poker values instead of embedding large lookup tables or model weights.

In [ ]:
%%writefile poker_bot.py
"""Edit act(state), export this .py file, and upload it at the host's /submit page.
Only your hole cards and public information are supplied. Return one legal action.
"""


def act(state):
    # Replace this simple baseline with your own poker algorithm.
    ranks = "23456789TJQKA"
    first, second = state["holeCards"]
    pair = first[0] == second[0]
    high_cards = ranks.index(first[0]) + ranks.index(second[0]) >= 19

    if state["street"] == "preflop" and state["canRaise"] and (pair or high_cards):
        return {"action": "raise", "raiseTo": min(state["minRaiseTo"], state["maxRaiseTo"])}

    # A free check is always legal. Calling costs at most your remaining stack.
    if state["toCall"] == 0:
        return {"action": "check"}
    price = min(state["toCall"], state["stack"]) / max(1, state["pot"] + min(state["toCall"], state["stack"]))
    if not pair and not high_cards and price > 0.25:
        return {"action": "fold"}
    return {"action": "call"}

## The decision contract

The server calls `act(state)` whenever it is your turn. Return an action; the server moves chips and deals cards.

| State field | Meaning |
| --- | --- |
| `holeCards` | Your two cards, e.g. `["As", "Kd"]` |
| `board`, `street` | Revealed community cards and `preflop` / `flop` / `turn` / `river` |
| `stack`, `bet`, `pot` | Available chips, your current-street contribution, and the pot |
| `toCall` | Chips required to match the current wager; may exceed your stack |
| `canRaise` | Whether raising is currently legal |
| `minRaiseTo`, `maxRaiseTo` | Legal raise targets; `maxRaiseTo` is your all-in total |
| `players`, `history` | Public opponents and recent actions; no opponent hole cards |

Cards use ranks `23456789TJQKA` and suits `c`, `d`, `h`, `s`. Examples:

```python
return {"action": "fold"}
return {"action": "check"}                    # Only when toCall == 0.
return {"action": "call"}                     # Pays min(toCall, stack).
return {"action": "raise", "raiseTo": 120}     # TOTAL current-street contribution.
```

If `bet` is 20, `raiseTo: 120` adds 100 chips. When raising is allowed, an all-in `maxRaiseTo` below `minRaiseTo` is also legal. Each decision gets a fresh namespace: use the supplied state rather than expecting Python globals to persist. Your bot cannot use hidden cards, the shuffle seed, model files, external AI services, or external algorithm libraries.

The host's `/submit` page links the full protocol. The notebook helpers below use extra libraries for testing and download; these helpers are **not part of `poker_bot.py`**.

## 2. Optional local smoke checks

This cell checks basic syntax, the entry-point signature, import names, and four sample decisions. It runs your own source in this notebook runtime, with a one-second timeout per subprocess.

**These are not a security sandbox, a complete server-policy check, or a playing-strength test.** Passing here does not approve your submission. The server independently enforces its full source policy, isolated runtime, legal-action checks, and 500 ms decision budget before accepting the file. Only run code you wrote or reviewed.

If a check fails, edit the bot cell, re-run it to overwrite `poker_bot.py`, then run this cell again.

In [ ]:
from pathlib import Path
import ast
import copy
import json
import subprocess
import sys

source_path = Path("poker_bot.py")
source = source_path.read_text(encoding="utf-8")
assert len(source.encode("utf-8")) <= 20000, "The source must be at most 20 KB."
tree = ast.parse(source, filename=str(source_path))
entries = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "act"]
assert len(entries) == 1, "Define exactly one top-level def act(state)."
args = entries[0].args
assert (len(args.args) == 1 and args.args[0].arg == "state" and not args.posonlyargs
        and not args.kwonlyargs and not args.vararg and not args.kwarg
        and not args.defaults and not args.kw_defaults), "Use exactly def act(state)."
approved_imports = {"math", "random", "itertools", "collections"}
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        assert all(item.name in approved_imports for item in node.names), "Only the four approved libraries may be imported."
    if isinstance(node, ast.ImportFrom):
        assert not node.level and node.module in approved_imports, "Only the four approved libraries may be imported."
        assert all(item.name != "*" for item in node.names), "Use explicit imports, not import *."

CASES = [{'name': 'Premium preflop', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 980, 'bet': 20, 'pot': 120, 'toCall': 20, 'currentBet': 40, 'minRaiseTo': 60, 'maxRaiseTo': 1000, 'canRaise': True, 'legalActions': ['fold', 'call', 'raise'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 960, 'bet': 40, 'contributed': 40, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}], 'street': 'preflop', 'holeCards': ['As', 'Ad'], 'board': []}}, {'name': 'Free flop check', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 970, 'bet': 0, 'pot': 150, 'toCall': 0, 'currentBet': 0, 'minRaiseTo': 20, 'maxRaiseTo': 970, 'canRaise': True, 'legalActions': ['check', 'fold', 'raise'], 'history': [], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 970, 'bet': 0, 'contributed': 30, 'folded': False, 'out': False, 'allIn': False}], 'street': 'flop', 'holeCards': ['7c', '2d'], 'board': ['Kh', '9s', '4c']}}, {'name': 'River facing a bet', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 900, 'bet': 0, 'pot': 700, 'toCall': 200, 'currentBet': 200, 'minRaiseTo': 400, 'maxRaiseTo': 900, 'canRaise': True, 'legalActions': ['fold', 'call', 'raise'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 700, 'bet': 200, 'contributed': 300, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 900, 'bet': 0, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}], 'street': 'river', 'holeCards': ['Ah', 'Jh'], 'board': ['Kh', 'Th', '2c', '4d', '8h']}}, {'name': 'Short stack call', 'state': {'schemaVersion': 1, 'hand': 1, 'seat': 0, 'dealer': 4, 'smallBlind': 10, 'bigBlind': 20, 'stack': 30, 'bet': 20, 'pot': 180, 'toCall': 80, 'currentBet': 100, 'minRaiseTo': 180, 'maxRaiseTo': 50, 'canRaise': False, 'legalActions': ['fold', 'call'], 'history': [{'seat': 1, 'street': 'preflop', 'action': 'raise', 'bet': 40}], 'players': [{'id': 0, 'name': 'Opponent1', 'seat': 0, 'stack': 30, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 1, 'name': 'Opponent2', 'seat': 1, 'stack': 900, 'bet': 100, 'contributed': 100, 'folded': False, 'out': False, 'allIn': False}, {'id': 2, 'name': 'Opponent3', 'seat': 2, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 3, 'name': 'Opponent4', 'seat': 3, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}, {'id': 4, 'name': 'Opponent5', 'seat': 4, 'stack': 980, 'bet': 20, 'contributed': 20, 'folded': False, 'out': False, 'allIn': False}], 'street': 'preflop', 'holeCards': ['Qs', 'Qd'], 'board': []}}]

# This test harness is outside the saved bot file. Each sample uses a fresh process.
runner = r"""
import contextlib
import json
import sys
with open(sys.argv[1], encoding="utf-8") as file:
    source = file.read()
state = json.loads(sys.stdin.read())
namespace = {"__name__": "student_bot"}
with contextlib.redirect_stdout(sys.stderr):
    exec(compile(source, sys.argv[1], "exec"), namespace)
    action = namespace["act"](state)
print(json.dumps(action, allow_nan=False))
"""

def check_action(action, state):
    assert isinstance(action, dict), "Return a dictionary."
    assert action.get("action") in {"fold", "check", "call", "raise"}, "Unknown or missing action."
    expected = {"action", "raiseTo"} if action["action"] == "raise" else {"action"}
    assert set(action) == expected, "Use only action and, for a raise, raiseTo."
    if action["action"] == "check":
        assert state["toCall"] == 0, "Cannot check while facing a bet."
    if action["action"] == "raise":
        target = action["raiseTo"]
        assert state["canRaise"], "Raising is not currently allowed."
        assert type(target) is int, "raiseTo must be an integer."
        assert state["currentBet"] < target <= state["maxRaiseTo"], "Raise target is outside your available chips."
        assert target >= state["minRaiseTo"] or target == state["maxRaiseTo"], "Raise too small unless it is all-in."

failures = []
for case in CASES:
    try:
        completed = subprocess.run(
            [sys.executable, "-I", "-B", "-c", runner, str(source_path.resolve())],
            input=json.dumps(copy.deepcopy(case["state"])), capture_output=True,
            text=True, timeout=1, check=True,
        )
        action = json.loads(completed.stdout)
        check_action(action, case["state"])
        print("PASS", case["name"], "→", action)
    except (AssertionError, ValueError, subprocess.SubprocessError) as error:
        detail = (getattr(error, "stderr", None) or str(error)).strip()
        failures.append(case["name"])
        print("FAIL", case["name"], "→", detail)

assert not failures, "Fix these local checks before uploading: " + ", ".join(failures)
print("\n4/4 local smoke checks passed. Upload the .py file for authoritative server validation.")

## 3. Download your submission

Run this cell after re-running the bot-writing cell with your final edits. Colab will download **`poker_bot.py`** to your computer. If prompted, allow the browser download. You can also find the file in Colab's Files sidebar and download it there.

The `google.colab` import belongs to this export helper only. Do not copy it into your submitted bot.

In [ ]:
from google.colab import files
files.download("poker_bot.py")

## 4. Upload, then let the server run it

Open the **student invitation link your host provided**. It opens the `/submit` page and supplies the event's class code. If the page asks for a class code, use the one your host gave you. Choose your entry name, upload `poker_bot.py`, accept the code rules, and wait for the server's validation result. If rejected, use the reported feedback to edit the function and upload again. Save the returned receipt/update key privately. To replace your entry, use the same name and its update key; each accepted revision is stored separately.

An accepted file joins the submitted roster; the host starts the tournament. Your Colab runtime does not need to stay connected during play. The server calls your uploaded algorithm, deals cards, enforces the rules, and displays the event. Existing events keep the source revision they started with.

Everyone plays the qualifiers. The top five (or all entrants in a three- or four-player event) start the final with equal stacks; the final's last surviving bot wins.

Official notebook helpers: [Saving a GitHub notebook to Drive](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb) · [Colab file downloads](https://colab.research.google.com/notebooks/io.ipynb) · [IPython writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile).